<a href="https://colab.research.google.com/github/EberHernandezBenitez/EDP/blob/main/Linea_espera_Ross.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Secciòn 6.2. Sistema de línea de espera con un servidor

Empleamos este código cuando queremos simular una línea de espera, en donde si no hay personas pasa directo y si hay personas, se mantiene en la línea dfe espera.\
Con $t=$ tiempo \
$N_A:$ el número de llegadas hasta el instante $t$ \
$N_D:$ el número de salidas hasta el instante $t$ \
$n:$ el número de clientes en el sistema
\
**Generamos dos eventos:** \
$t_A:$ tiempo de la siguiente llegada.
\
$t_D:$ tiempo de la siguiente salida
\
**Queremos:**
\
a) Tiempo promedio de un cliente en el sistema:.
\
b) Tiempo en que sale el último cliente.


Para ello, generamos $T_0 \to U~U(0,1)$\
$t= t-\frac{1}{\lambda}logU$\
$t_A=T_0$\
$t_D= ∞$

In [ ]:
import math
import random as r


def generar_tiempo_llegada(t_actual, lam):
    rand_num = r.random()
    return t_actual - (math.log(rand_num) / lam)


def generar_tiempo_servicio(mu):
    # Genera la variable aleatoria Y (tiempo de servicio) con distribución exponencial
    rand_num = r.random()
    return -(math.log(rand_num) / mu)


def simular_linea_espera(lam, mu, T):
    t = 0.0
    N_A = 0
    N_D = 0
    n = 0

    # Listas para recopilar datos de salida
    A = {}  # Registra A(i): hora de llegada del cliente i
    D = {}  # Registra D(i): hora de salida del cliente i

    t_A = generar_tiempo_llegada(t, lam)
    t_D = float("inf")  # Al principio no hay nadie en servicio

    while True:
        # Caso 1: El próximo evento es una llegada y ocurre antes del tiempo límite T
        if t_A <= t_D and t_A <= T:
            t = t_A
            N_A += 1
            n += 1

            # Generar la hora de la siguiente llegada
            t_A = generar_tiempo_llegada(t, lam)

            # Si el cliente que llegó encontró el sistema vacío, pasa directo a servicio
            if n == 1:
                Y = generar_tiempo_servicio(mu)
                t_D = t + Y

            # Guardar datos de salida de la llegada
            A[N_A] = t

        # Caso 2: El próximo evento es una salida y o curre antes o en el tiempo límite T
        elif t_D < t_A and t_D <= T:
            t = t_D  # Avanzar el tiempo
            n -= 1  # El cliente sale del sistema
            N_D += 1  # Incrementar contador de salidas

            # Guardar datos de salida del cliente que se va
            D[N_D] = t

            # Si quedan clientes en fila, el siguiente pasa a ser atendido
            if n == 0:
                t_D = float("inf")
            else:
                Y = generar_tiempo_servicio(mu)
                t_D = t + Y

        # Caso 3: Ya pasó el tiempo límite T, pero todavía quedan clientes en el sistema
        elif min(t_A, t_D) > T and n > 0:
            t = t_D  # Avanzar el tiempo al de la salida
            n -= 1  # El cliente sale del sistema
            N_D += 1  # Incrementar contador de salidas

            # Guardar datos de salida
            D[N_D] = t

            # Si todavía quedan más clientes, atender al siguiente
            if n > 0:
                Y = generar_tiempo_servicio(mu)
                t_D = t + Y

        # Caso 4: Ya pasó el tiempo límite T y el sistema se ha vaciado por completo
        elif min(t_A, t_D) > T and n == 0:
            # Calcular el tiempo extra que el servidor se quedó después de T
            T_p = max(t - T, 0)
            break  # Terminar la ejecución de la simulación

    # (a) Tiempo promedio que pasa un cliente dentro del sistema (D(i) - A(i))
    tiempos_en_sistema = [D[i] - A[i] for i in range(1, N_D + 1)]
    tiempo_promedio_sistema = (
        sum(tiempos_en_sistema) / N_D if N_D > 0 else 0.0
    )

    return {
        "N_A": N_A,
        "N_D": N_D,
        "Tiempo_Promedio_Sistema": tiempo_promedio_sistema,
        "T_p": T_p,
    }


# Parámetros para el ejercicio: lambda = 4, mu = 6. Definimos un tiempo de corte T = 100
LAMBDA_VAL = 4
MU_VAL = 6
TIEMPO_CORTE = 100

resultado = simular_linea_espera(LAMBDA_VAL, MU_VAL, TIEMPO_CORTE)

print("=== RESULTADOS DE LA SIMULACIÓN ===")
print(f"Total de clientes que llegaron (N_A): {resultado['N_A']}")
print(f"Total de clientes que salieron (N_D): {resultado['N_D']}")
print(
    f"Tiempo promedio de un cliente en el sistema: {resultado['Tiempo_Promedio_Sistema']:.4f}"
)
print(
    f"Tiempo extra del servidor después de T (T_p): {resultado['T_p']:.4f}"
)

=== RESULTADOS DE LA SIMULACIÓN ===
Total de clientes que llegaron (N_A): 385
Total de clientes que salieron (N_D): 385
Tiempo promedio de un cliente en el sistema: 0.4837
Tiempo extra del servidor después de T (T_p): 0.8882


Analiticamente podemos calcular el **tiempo promedio en el sistema(espera+atención)** con $\mu=6$ y $\lambda=4$:

$$W_s=W_q+\frac{1}{\mu}=\frac{1}{\mu-\lambda}$$

$$W_s=\frac{1}{\mu-\lambda}=\frac{1}{6-4}=\frac{1}{2}$$
Como vemos, en la simulación el tiempo promedio en el sistema nos dan valores cercanos a **$\frac{1}{2}$** por lo que el error no es muy grande.